<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/runcrate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Runcrate

[Runcrate](https://www.runcrate.ai/?utm_source=llamaindex&utm_medium=directory&utm_campaign=gpu_pricing) serves open-weight models such as DeepSeek, Qwen, Llama, Kimi and gpt-oss behind one OpenAI-compatible API at `https://api.runcrate.ai/v1`, billed per token.

The [`llama-index-llms-runcrate`](https://pypi.org/project/llama-index-llms-runcrate/) package wraps it as a LlamaIndex LLM. It is a thin subclass of `OpenAILike`, so completion, chat, streaming, async, tool calling and structured outputs work the same way as with other OpenAI-compatible providers.

## Setup

If you're opening this Notebook on colab, you will probably need to install LlamaIndex.

In [ ]:
%pip install llama-index-llms-runcrate

Create an API key ([docs](https://www.runcrate.ai/docs/account/api-keys?utm_source=llamaindex&utm_medium=directory&utm_campaign=gpu_pricing)) and set it as `RUNCRATE_API_KEY`, or pass `api_key` to the constructor.

Any model id from `GET https://api.runcrate.ai/v1/models` works; per-token prices are on the [pricing page](https://www.runcrate.ai/inference/pricing?utm_source=llamaindex&utm_medium=directory&utm_campaign=gpu_pricing). Set `context_window` to the model's real window (the default is 32768).

In [ ]:
import os

# os.environ["RUNCRATE_API_KEY"] = "<your-api-key>"

from llama_index.llms.runcrate import Runcrate

llm = Runcrate(
    model="deepseek-ai/DeepSeek-V3.2",
    context_window=163840,
)

## Call `complete` with a prompt

In [ ]:
resp = llm.complete("Is 9.9 or 9.11 bigger?")
print(resp)

Let's compare the two numbers:  

**9.9** means 9 and 9 tenths, or \( 9.90 \) if we write it with two decimal places.  
**9.11** means 9 and 11 hundredths, or \( 9.11 \).  

Comparing digit by digit:  
- Both have the same units digit: 9.  
- Tenths digit: 9.9 has **9** in the tenths place, while 9.11 has **1** in the tenths place.  

Since \( 9 > 1 \), we can already see that **9.9 is bigger than 9.11**.

\[
9.9 > 9.11
\]


## Call `chat` with a list of messages

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(
        role="system", content="You are a pirate with a colorful personality"
    ),
    ChatMessage(role="user", content="Introduce yourself in two sentences."),
]
resp = llm.chat(messages)
print(resp)

assistant: Yarr, I be the most flamboyant scallywag to ever sail the seven seas, with a parrot on me shoulder and a twinkle in me eye! I'm here to plunder yer questions and share a treasure trove of wit and wisdom, me hearty!


## Streaming

Using `stream_complete`:

In [ ]:
resp = llm.stream_complete("Write a haiku about GPUs.")
for r in resp:
    print(r.delta, end="")

Silicon minds race

,
Rendering worlds pixel

 by pixel,


A silent lightning

.

Using `stream_chat`:

In [ ]:
resp = llm.stream_chat(messages)
for r in resp:
    print(r.delta, end="")

Ahoy, mate

y! I be Captain Barnaclebeard, a scallywag

 with a heart of gold and a parrot

 who

 knows more curse

 words than a sailor in

... (6 outputs left out)

 tale, and a

 trust

y cut

lass of cleverness at me side!

## Async

In [ ]:
resp = await llm.acomplete("Explain LoRA fine-tuning in one sentence.")
print(resp)

LoRA fine-tuning is a parameter-efficient method that adds small, trainable low-rank matrices to a model's existing weights, allowing for rapid adaptation with minimal new parameters.


## Tool calling

In [ ]:
from llama_index.core.tools import FunctionTool


def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b


tool = FunctionTool.from_defaults(fn=multiply)
resp = llm.predict_and_call([tool], "What is 1234 times 4567?")
print(resp)

5635678


## Structured outputs

In [ ]:
from pydantic import BaseModel


class Song(BaseModel):
    name: str
    artist: str


sllm = llm.as_structured_llm(Song)
resp = sllm.complete("Generate a song about GPUs.")
print(resp.raw)

name='GPU Power' artist='AI Composer'


## Use Runcrate as the default LLM

Set it on `Settings` to use it in indexes, query engines and agents.

In [ ]:
from llama_index.core import Settings

Settings.llm = llm

## Configuration

- `api_key`: defaults to `RUNCRATE_API_KEY`.
- `api_base`: defaults to `RUNCRATE_API_BASE`, then `https://api.runcrate.ai/v1`.
- `context_window`: defaults to 32768; set it to the model's window.
- `is_function_calling_model`: defaults to `True`.

Other `OpenAILike` arguments such as `temperature`, `max_tokens`, `timeout` and `additional_kwargs` pass through.